In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
df = pd.read_excel(
    "../../../data/hospital_billing.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "isCancelled": "string",
        "isClosed": "string",
        "caseType": "string",
        "speciality": "string",
        "blocked": "string",
        "flagD": "string",
        "flagB": "string",
        "flagA": "string",
        "state": "string",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,blocked,caseType,concept:name,flagA,flagB,flagD,isCancelled,isClosed,lifecycle:transition,speciality,state,time_delta
0,A,2012-12-16 19:33:10,False,A,NEW,False,False,True,False,True,complete,A,In progress,0.0
1,A,2013-12-15 19:00:37,NA,NA,FIN,NA,NA,NA,NA,NA,complete,NA,Closed,31447648.0
2,A,2013-12-16 03:53:38,NA,NA,RELEASE,NA,NA,NA,NA,NA,complete,NA,Released,31981.0
3,A,2013-12-17 12:56:29,NA,NA,CODE OK,NA,NA,NA,NA,NA,complete,NA,NA,118971.0
4,A,2013-12-19 03:44:31,NA,NA,BILLED,NA,NA,NA,NA,NA,complete,NA,Billed,139682.0
5,AA,2012-12-26 08:50:18,False,B,NEW,False,False,True,False,True,complete,L,In progress,0.0
6,AA,2012-12-26 08:50:59,NA,NA,CHANGE DIAGN,NA,NA,NA,NA,NA,complete,NA,In progress,41.0
7,AA,2013-02-14 21:06:33,NA,NA,FIN,NA,NA,NA,NA,NA,complete,NA,Closed,4364134.0
8,AA,2013-02-14 22:12:10,NA,NA,RELEASE,NA,NA,NA,NA,NA,complete,NA,Released,3937.0
9,AA,2013-02-18 01:44:10,NA,NA,CODE OK,NA,NA,NA,NA,NA,complete,NA,NA,271920.0


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['blocked', 'caseType', 'concept:name', 'flagA', 'flagB', 'flagD', 'isCancelled', 'isClosed', 'lifecycle:transition', 'speciality', 'state', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
isCancelled                    categorical    event    yes    ['False', 'True']                        N/A        data_derived        
isClosed                       categorical    event    yes    ['False', 'True']                        N/A        data_derived        
caseType                       categorical    event    yes    ['A', 'B', 'C', ...]                     N/A     

In [7]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [8]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [9]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [10]:
scenario_df.head()

,case:concept:name,time_index,fake,blocked,caseType,concept:name,flagA,flagB,flagD,isCancelled,isClosed,lifecycle:transition,speciality,state,time_delta
0,GAXD,0,False,False,I,NEW,False,False,False,False,True,complete,K,In progress,0.0
1,FKCC,0,False,False,B,NEW,False,False,True,False,True,complete,C,In progress,0.0
2,FKCC,1,False,NA,NA,CHANGE DIAGN,NA,NA,NA,NA,NA,complete,NA,In progress,1198180.0
3,FKCC,2,False,NA,NA,FIN,NA,NA,NA,NA,NA,complete,NA,Closed,5152082.0
4,FKCC,3,False,NA,NA,RELEASE,NA,NA,NA,NA,NA,complete,NA,Released,20079.0


In [11]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [12]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [13]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
criterion = torch.nn.BCEWithLogitsLoss()

In [16]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/hospital_billing-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0001 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0001 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0001 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0000 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0000 | LR: 1.00e-06
Time taken for scenario model (training): 38398.479972 seconds
Time taken for scenario model (validation): 26.918051 seconds
Val loss: {'loss': 0.00011195036198996804, 'accuracy': 0.9999861371424769, 'f1_macro': 0.9999815105155121, 'f1_weighted': 0.9999861371168334}


In [17]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [18]:
scenario_model = ScenarioLSTM.load()

In [19]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [20]:
sys.stdout = original_stdout
log_file.close()